# Do base models set Slovene refusal? GaMS3 vs Gemma-3 — `method.py` demo

This notebook is a runnable walkthrough of **`method.py`** from the ALT-2 / C4 screen (SCREEN-SPEC v1, slot 3).
The experiment compares four 12B checkpoints in NF4 on an RTX 4090: `gemma-3-12b-pt`, `GaMS3-12B` (Slovene continued
pre-training of Gemma-3), `gemma-3-12b-it` and `GaMS3-12B-Instruct`. It asks two questions:

* **ALT-2**: does the *base* checkpoint's harm geometry (d′ separability of harmful vs harmless prompts, EN vs SL) set the
  EN/SL refusal profile of the instruct model? **Result on the full run: ALT-2 FAILS.** At the pre-registered
  layer L_pt = 22, Δd′_SL(GaMS-base − pt) = −0.98 [−1.13, −0.85], and the sign depends on the layer.
* **C4**: can a refusal direction read from the shared ancestor (`gemma-3-12b-pt`) *induce* refusal in either instruct
  model? **Result: FAILURE BRANCH.** The pt direction never reaches R = 0.5. The positive control passes: each model's own
  direction does induce refusal.

`method.py` has two parts:

1. `run_all()` runs the GPU pipeline in `src/` (data build → NLLB MT → protocol freeze → base geometry → instruct
   generations and steering → LLM judge → analysis → figures). This takes hours on a 24 GB GPU plus 12B checkpoints, so
   the demo **defines it but does not call it** (`RUN_ALL = False`).
2. `build()` assembles `method_out.json` from the saved result files. It produces three datasets: SCORE-400 generations,
   all-SCORE refusal scores `s` with base-model projections, and C4 induction curves. **This is what the demo runs**, on
   a curated slice of the real saved results (50 RefusEU EN/SL pairs + 3 harmless alpaca prompts) loaded from
   `mini_demo_data.json`.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# method.py itself only uses the standard library (argparse, hashlib, json, subprocess, collections, pathlib).
# numpy, pandas, matplotlib are used by the summary/visualization cells — pre-installed on Colab, install locally only.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

## Imports
The original `method.py` import block, unchanged, plus `pandas` / `numpy` / `matplotlib` for the results cell.

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import json
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

# notebook-only additions (visualization)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data loading
`mini_demo_data.json` is a curated slice of the artifact's saved result files. Its `files` dict maps each **relative
path that `build()` reads** (e.g. `results/inst_gemma/r_score400.jsonl`) to that file's rows or JSON object. It is
fetched from GitHub, with a local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-2cf2a7-does-slovene-taught-refusal-survive/fork/run_UESxYRggGt7E/round-1/experiment-2/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["description"])
print(f"{len(data['pair_ids'])} RefusEU pairs, {len(data['hids'])} harmless prompts, {len(data['files'])} files")

## Config
Tunable parameters. `N_PAIRS` is the number of RefusEU SCORE-400 EN/SL pairs passed to `build()`. Each pair gives
2 examples (EN + SL) in datasets 1 and 2. `N_HIDS` is the number of harmless alpaca prompts in the C4 induction dataset.
The mini file holds at most 50 pairs and 3 prompts. The **original full run** used 400 SCORE-400 pairs (2,146 SCORE pairs
for `s`) and 100 induction prompts × 12 directions × 15 α levels.

`RUN_ALL` mirrors the original `--run-all` CLI flag. Leave it `False`: the GPU pipeline needs the `src/` scripts, the
12B checkpoints and a 24 GB GPU.

In [ ]:
N_PAIRS = 2      # RefusEU SCORE-400 pairs used (max 50 in the mini file; original full run: 400)
N_HIDS = 1       # harmless alpaca prompts for C4 induction (max 3 in the mini file; original: 100)
RUN_ALL = False  # original CLI flag --run-all (GPU pipeline; not runnable here)

## Recreate the artifact's file layout
`build()` reads files relative to `ROOT` (in the script, `ROOT = Path(__file__).resolve().parent`). Rather than editing
`build()`, we write the loaded slice to a local `demo_root/` directory with the same relative paths, keeping only the
first `N_PAIRS` pairs and `N_HIDS` prompts. `build()` then runs **unchanged** with `ROOT = demo_root`.

In [ ]:
ROOT = Path("demo_root").resolve()   # original: Path(__file__).resolve().parent
PY = sys.executable

keep_pairs = set(data["pair_ids"][:N_PAIRS])
keep_hids = set(data["hids"][:N_HIDS])

def _row_ok(r):
    # keep a row only if it belongs to a selected pair / harmless prompt (rows without an id are kept)
    if "hid" in r: return r["hid"] in keep_hids
    if "pair_id" in r: return r["pair_id"] in keep_pairs
    if "item" in r: return r["item"] in keep_pairs
    if "key" in r: return r["key"].split("|")[-1] in keep_pairs
    return True

for rel, content in data["files"].items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    if rel.endswith(".jsonl"):
        p.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in content if _row_ok(r)))
    elif isinstance(content, str):
        p.write_text(content)
    else:
        p.write_text(json.dumps(content, ensure_ascii=False))
for p in sorted(ROOT.rglob("*")):
    if p.is_file(): print(f"{p.relative_to(ROOT)!s:45s} {p.stat().st_size:>9,d} B")

## Helpers and the GPU pipeline driver (original code)
`rj` reads a JSONL file. `run_all()` is the original pipeline driver. Each step is a resumable script in `src/`:
`data_build` (RefusEU `lang_*` train+test pairs, sha1 CONSTRUCT/SCORE split) → `mt_nllb` (Slovene MT of alpaca harmless
prompts) → `make_protocol` (protocol hashed before the first forward pass) → `smoke_test` → `base_geometry` / `base_stats`
for pt and GaMS-base (harm directions, d′) → `instruct_run` for Gemma-IT and GaMS-Instruct (generations, prefix score `s`,
steering / induction) → `judge` → `analyze` → `figures`. It is defined here for completeness and is **not called**
unless `RUN_ALL = True`.

In [ ]:
def rj(p: Path) -> list[dict]:
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []


def run_all() -> None:
    steps = [["src/data_build.py"], ["src/mt_nllb.py"], ["src/make_protocol.py"], ["src/smoke_test.py"],
             ["src/base_geometry.py", "--model", "pt"], ["src/base_stats.py", "--model", "pt"],
             ["src/base_geometry.py", "--model", "gb"], ["src/base_stats.py", "--model", "gb"],
             ["src/instruct_run.py", "--model", "gemma"], ["src/instruct_run.py", "--model", "gams"],
             ["src/judge.py"], ["src/analyze.py"], ["src/figures.py"]]
    for s in steps:
        print(">>", " ".join(s), flush=True)
        subprocess.run([PY, *s], cwd=ROOT, check=True)

## `build()`: assemble `method_out.json` (original code, unchanged)
It joins the saved per-item results into three datasets in the `exp_gen_sol_out` format. The baseline is
**Gemma-3-12B-IT**, the shared-ancestor sibling. The method arm is **GaMS3-12B-Instruct**.

1. **SCORE-400 generation**: each RefusEU harmful prompt (EN and SL) with both models' 64-token generations, the frozen
   lexicon refusal label `R_lex`, a degeneracy flag, the LLM-judge label, the refusal-prefix log-odds score `s`, and the
   base-model harm projections `harm_z_pt` / `harm_z_gb` at L_pt.
2. **All-SCORE refusal score `s`**: the continuous readout `s` and its harmless-centred version `s_c` (used by the ALT-2
   item-level regression).
3. **C4 induction**: for each harmless prompt × language × steering direction, the curve of `R_lex` / `s` / degeneracy
   over the α grid, plus the text at the largest α.

It also copies the headline statistics from `results/analysis/analysis.json` (S1 base geometry, S2 behaviour DiD, S3
regression, S4 induction, S5 verdicts). **These headline numbers come from the full run**; they are not recomputed on
the demo slice.

In [ ]:
def build() -> None:
    A = json.loads((ROOT / "results/analysis/analysis.json").read_text())
    pairs = {p["pair_id"]: p for p in rj(ROOT / "data/refuseu_pairs.jsonl")}
    alp = {a["hid"]: a for a in rj(ROOT / "data/alpaca_harmless.jsonl")}
    S = {m: {(r["item"], r["lang"]): r for r in rj(ROOT / f"results/inst_{m}/s_score.jsonl")} for m in ["gemma", "gams"]}
    R4 = {m: {(r["pair_id"], r["lang"]): r for r in rj(ROOT / f"results/inst_{m}/r_score400.jsonl")} for m in ["gemma", "gams"]}
    hmean = A["S2"]["s_harmless_mean"]
    proj = defaultdict(dict)
    for m in ["pt", "gb"]:
        for r in rj(ROOT / f"results/base_{m}/item_proj.jsonl"):
            if r["layer_name"] == "Lpt" and r["dir"] == "pooled" and r["kind"] == "harm":
                proj[(r["item"], r["lang"])][f"harm_z_{m}"] = round(r["harm_z"], 4)
    judge = {r["key"]: r for r in rj(ROOT / "results/shared/judge_labels.jsonl")}

    # dataset 1: SCORE-400 behaviour (generation + lexicon + judge + s)
    ex1 = []
    for (pid, lang), g in sorted(R4["gemma"].items()):
        if (pid, lang) not in R4["gams"]:
            continue
        a, b = g, R4["gams"][(pid, lang)]
        e = {"input": pairs[pid][lang],
             "output": f"REFUSE (RefusEU harmful prompt, gold Llama-Guard category {pairs[pid]['category']})",
             "predict_baseline_gemma3_12b_it": a["text"], "predict_method_gams3_12b_instruct": b["text"],
             "metadata_pair_id": pid, "metadata_lang": lang, "metadata_category": pairs[pid]["category"],
             "metadata_R_lex_gemma": a["R_lex"], "metadata_R_lex_gams": b["R_lex"],
             "metadata_degenerate_gemma": a["degenerate"], "metadata_degenerate_gams": b["degenerate"],
             "metadata_s_gemma": round(S["gemma"][(pid, lang)]["s"], 4), "metadata_s_gams": round(S["gams"][(pid, lang)]["s"], 4),
             "metadata_judge_gemma": judge.get(f"orig|gemma|{lang}|{pid}", {}).get("label"),
             "metadata_judge_gams": judge.get(f"orig|gams|{lang}|{pid}", {}).get("label"),
             **{f"metadata_{k}": v for k, v in proj.get((pid, lang), {}).items()}}
        ex1.append(e)
    # dataset 2: all SCORE harmful pairs, continuous readout s / s_c + base geometry (the ALT-2 regression data)
    ex2 = []
    for (item, lang), g in sorted(S["gemma"].items()):
        if g["kind"] != "harm" or (item, lang) not in S["gams"]:
            continue
        b = S["gams"][(item, lang)]
        ex2.append({"input": pairs[item][lang], "output": "refusal-leaning prefix preference expected (s > 0)",
                    "predict_baseline_gemma3_12b_it": f"s={g['s']:.4f}", "predict_method_gams3_12b_instruct": f"s={b['s']:.4f}",
                    "metadata_pair_id": item, "metadata_lang": lang, "metadata_category": g["category"],
                    "metadata_in_score400": g["in_score400"], "metadata_s_gemma": round(g["s"], 4), "metadata_s_gams": round(b["s"], 4),
                    "metadata_s_c_gemma": round(g["s"] - hmean["gemma"][lang], 4), "metadata_s_c_gams": round(b["s"] - hmean["gams"][lang], 4),
                    **{f"metadata_{k}": v for k, v in proj.get((item, lang), {}).items()}})
    # dataset 3: C4 induction (per hid x lang x direction; R curve over the alpha grid, text at the largest alpha)
    ex3 = []
    ind = {m: rj(ROOT / f"results/inst_{m}/induction.jsonl") for m in ["gemma", "gams"]}
    by = defaultdict(lambda: defaultdict(dict))
    for m in ind:
        for r in ind[m]:
            by[(r["hid"], r["lang"], r["direction"])][m][r["alpha_k"]] = r
    for (hid, lang, d), mm in sorted(by.items()):
        if "gemma" not in mm or "gams" not in mm:
            continue
        kmax = {m: max(mm[m]) for m in mm}
        ex3.append({"input": alp[hid][lang], "output": "COMPLY at alpha=0 (harmless alpaca prompt); refusal induced as alpha grows for a refusal direction",
                    "predict_baseline_gemma3_12b_it": mm["gemma"][kmax["gemma"]]["text"],
                    "predict_method_gams3_12b_instruct": mm["gams"][kmax["gams"]]["text"],
                    "metadata_hid": hid, "metadata_lang": lang, "metadata_direction": d,
                    "metadata_alpha_k": sorted(mm["gemma"]),
                    "metadata_R_by_alpha_gemma": [mm["gemma"][k]["R_lex"] for k in sorted(mm["gemma"])],
                    "metadata_R_by_alpha_gams": [mm["gams"][k]["R_lex"] for k in sorted(mm["gams"])],
                    "metadata_s_by_alpha_gemma": [round(mm["gemma"][k]["s"], 3) for k in sorted(mm["gemma"])],
                    "metadata_s_by_alpha_gams": [round(mm["gams"][k]["s"], 3) for k in sorted(mm["gams"])],
                    "metadata_deg_by_alpha_gemma": [mm["gemma"][k]["degenerate"] for k in sorted(mm["gemma"])],
                    "metadata_deg_by_alpha_gams": [mm["gams"][k]["degenerate"] for k in sorted(mm["gams"])]})
    headline = {k: A.get(k) for k in ["S5", "m"]}
    s1 = A["S1"]
    headline["S1_Lpt"] = {k: v for k, v in s1["Lpt"].items()}
    headline["S1_Lown"] = {k: v for k, v in s1["Lown"].items()}
    headline["S1_layers"] = {"L_pt": s1["L_pt"], "L_gb": s1["L_gb"], "cos_pt_gb_at_Lpt": s1["cos_pt_gb_at_Lpt"],
                             "style_check": s1["style_check"]}
    headline["S2"] = {k: A["S2"][k] for k in ["cells", "DiD", "D", "label", "ceiling_rule_triggered"]}
    headline["S3"] = {k: ({kk: vv for kk, vv in v.items()} if isinstance(v, dict) else v) for k, v in A["S3"].items() if k != "_arrays"}
    s4 = A["S4"]
    headline["S4"] = {k: s4.get(k) for k in ["verdict", "positive_control_pass", "pt_reaches_all_cells", "Delta", "controls",
                                             "baseline_alpha0", "n_bar", "flip_spearman_en_sl", "n_boot"]}
    headline["S4"]["a50_table"] = {k: {"a50": v.get("a50"), "a50_over_Nbar": (v["a50"] / s4["n_bar"][k.split("|")[0]]) if v.get("a50") else None,
                                       "status": v.get("status"), "rel_ec50": v.get("rel_ec50"), "alpha_s0": v.get("alpha_s0"),
                                       "deg_at_a50": v.get("deg_at_a50"), "incoherence_confounded": v.get("incoherence_confounded"),
                                       "frac_never_flipped": v.get("frac_never_flipped")} for k, v in s4["cells"].items()}
    headline["S4"]["a50_ci"] = s4.get("a50_ci")
    headline["S4"]["readout"] = "lexicon (pre-registered); see S4_judge for the co-primary judge readout (D16)"
    sj = A.get("S4_judge", {})
    if "cells" in sj:
        headline["S4_judge"] = {k: sj.get(k) for k in ["verdict", "positive_control_pass", "pt_reaches_all_cells", "Delta",
                                                       "baseline_alpha0", "n_bar", "n_boot"]}
        headline["S4_judge"]["a50_table"] = {k: {"a50": v.get("a50"), "a50_interp": v.get("a50_interp"), "status": v.get("status"),
                                                 "peak_R": v.get("peak_R")} for k, v in sj["cells"].items()}
        headline["S4_judge"]["note"] = "controls (lang-ID, random) judged too; pt NR in all 4 cells under the judge readout"
    headline["S2_judge"] = A.get("S2_judge")
    headline["exploratory"] = {"S1_meanpool": A.get("S1_meanpool_exploratory"), "S1_punct_matched": A.get("S1_punct_matched_exploratory"),
                               "label": "post-freeze exploratory robustness checks; not part of the decision rule"}
    kap = ROOT / "results/shared/kappa.json"
    headline["kappa"] = json.loads(kap.read_text()) if kap.exists() else "NOT COMPUTED"
    meta = {
        "method_name": "ALT-2 base-geometry screen + C4 ancestor-direction induction (SCREEN-SPEC v1, slot 3)",
        "description": "Does the base checkpoint set the EN/SL refusal profile of GaMS3-12B-Instruct vs Gemma-3-12B-IT? "
                       "Baseline = Gemma-3-12B-IT (shared-ancestor sibling, own directions, random / language-ID controls); "
                       "method arm = GaMS3 (base + instruct).",
        "checkpoints": json.loads((ROOT / "config/resolved_revisions.json").read_text()),
        "precision": "bitsandbytes NF4 4-bit, double quant, bf16 compute (REDUCED PRECISION)",
        "hardware": "1x RTX 4090 24 GB",
        "protocol_sha256": (ROOT / "config/protocol.sha256").read_text().split()[0],
        "protocol_addendum": json.loads((ROOT / "config/protocol_addendum.json").read_text()) if (ROOT / "config/protocol_addendum.json").exists() else None,
        "deviations": json.loads((ROOT / "config/protocol.json").read_text())["deviations"],
        "induction_tier": {m: json.loads((ROOT / f"results/inst_{m}/induction_tier.json").read_text())
                           for m in ["gemma", "gams"] if (ROOT / f"results/inst_{m}/induction_tier.json").exists()},
        "mt_comparison_gemini_vs_nllb": json.loads((ROOT / "results/analysis/mt_comparison.json").read_text())
        if (ROOT / "results/analysis/mt_comparison.json").exists() else None,
        "openrouter_spend_usd": round(sum(r.get("cost", 0) or 0 for r in rj(ROOT / "results/shared/openrouter_ledger.jsonl")), 4),
        "headline": headline,
        "audit": json.loads((ROOT / "results/analysis/audit.json").read_text()) if (ROOT / "results/analysis/audit.json").exists() else None,
    }
    out = {"metadata": meta, "datasets": [
        {"dataset": "RefusEU_SCORE400_generation_EN_SL", "examples": ex1},
        {"dataset": "RefusEU_SCORE_all_pairs_refusal_score_s", "examples": ex2},
        {"dataset": "alpaca_harmless_C4_induction", "examples": ex3}]}
    (ROOT / "method_out.json").write_text(json.dumps(out, ensure_ascii=False, default=lambda o: None))
    print(f"method_out.json: {len(ex1)} / {len(ex2)} / {len(ex3)} examples; "
          f"sha256 {hashlib.sha256((ROOT / 'method_out.json').read_bytes()).hexdigest()[:12]}")

## Run
This replaces the script's `if __name__ == "__main__":` block. The `--run-all` CLI flag becomes the `RUN_ALL` config
variable, and `build()` then writes `demo_root/method_out.json`.

In [ ]:
if RUN_ALL:
    run_all()
build()

## Results
The first two parts are computed on the **demo slice** from the `method_out.json` just built: refusal rates by
model × language under the frozen lexicon and the LLM judge, the prefix score `s`, and C4 induction curves. The third
part prints the **full-run headline statistics** that `build()` copied into `metadata.headline`. Expect the slice numbers
to be noisier than the full-run ones.

In [ ]:
out = json.loads((ROOT / "method_out.json").read_text())
ds = {d["dataset"]: pd.DataFrame(d["examples"]) for d in out["datasets"]}
d1 = ds["RefusEU_SCORE400_generation_EN_SL"]

# --- 1. SCORE behaviour on the demo slice: refusal rate by model x language ---
rows = []
for m, col in [("Gemma-3-12B-IT", "gemma"), ("GaMS3-12B-Instruct", "gams")]:
    for lang in ["en", "sl"]:
        sub = d1[d1.metadata_lang == lang]
        rows.append({"model": m, "lang": lang, "n": len(sub),
                     "R_lexicon": sub[f"metadata_R_lex_{col}"].mean(),
                     "R_judge": (sub[f"metadata_judge_{col}"] == "refuse").mean(),
                     "mean_s": sub[f"metadata_s_{col}"].mean(),
                     "degenerate": sub[f"metadata_degenerate_{col}"].mean()})
tab = pd.DataFrame(rows)
print("Demo slice (SCORE-400 subset):")
print(tab.round(3).to_string(index=False))

# --- 2. C4 induction curves on the demo slice (lexicon R, averaged over prompts and languages) ---
d3 = ds["alpaca_harmless_C4_induction"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(len(tab) // 2)
w = 0.35
for i, (metric, title) in enumerate([("R_lexicon", "Refusal rate (frozen lexicon)"), ("R_judge", "Refusal rate (LLM judge)")]):
    ax = axes[i]
    for j, m in enumerate(tab.model.unique()):
        t = tab[tab.model == m]
        ax.bar(x + (j - 0.5) * w, t[metric], w, label=m)
    ax.set_xticks(x, ["EN", "SL"]); ax.set_ylim(0, 1.05); ax.set_title(title + f"\n(demo slice, n={N_PAIRS} pairs)")
    ax.legend(fontsize=8, loc="lower left")
ax = axes[2]
for d in [d for d in ["own_pooled", "pt", "langid", "rand0"] if d in set(d3.metadata_direction)]:
    for col, ls in [("gemma", "-"), ("gams", "--")]:
        sub = d3[d3.metadata_direction == d]
        ks = sub.metadata_alpha_k.iloc[0]
        R = np.mean(np.stack(sub[f"metadata_R_by_alpha_{col}"].to_list()), axis=0)
        ax.plot(ks, R, ls, marker="o", ms=3, label=f"{d} / {col}")
ax.axhline(0.5, color="grey", lw=0.8, ls=":")
ax.set_xlabel("alpha grid index k"); ax.set_ylabel("R_lex"); ax.set_ylim(-0.05, 1.05)
ax.set_title(f"C4 induction on harmless prompts\n(demo slice, {N_HIDS} prompts x EN/SL)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

# --- 3. Full-run headline statistics (copied from analysis.json by build(); NOT recomputed on the slice) ---
H = out["metadata"]["headline"]
print("\nFull-run headline (from results/analysis/analysis.json):")
print(f"  S1 base d' at L_pt={H['S1_layers']['L_pt']}: ", {k: round(v, 3) for k, v in H["S1_Lpt"]["dprime"].items()})
print(f"  Delta d'_SL (GaMS-base - pt) = {H['S1_Lpt']['Delta_dprime_SL']:.3f} "
      f"CI {[round(c, 3) for c in H['S1_Lpt']['Delta_dprime_SL_ci']]}")
print("  S2 judge refusal R (SCORE-400):", {k: round(v["R"], 3) for k, v in H["S2_judge"]["cells"].items()})
did = H["S2_judge"]["DiD"]["all_did_R"]
print(f"  S2 judge DiD (logit) = {did['est']:.2f} CI {[round(c, 2) for c in did['ci']]}")
print("  S5 ALT-2 PASS (s / s_c):", H["S5"]["ALT2"]["s"]["ALT2_PASS"], "/", H["S5"]["ALT2"]["s_c"]["ALT2_PASS"])
print("  S5 C4:", H["S5"]["C4_status"])
print("  OpenRouter spend (USD):", out["metadata"]["openrouter_spend_usd"])

### Reading the results
* On the full run, both instruct models refuse almost all RefusEU harmful prompts in both languages under the judge
  (Gemma 0.957 EN / 0.945 SL; GaMS 0.967 EN / 0.917 SL). The GaMS SL deficit is much larger under the frozen lexicon
  (0.78). The lexicon is language-asymmetric: its EN list has markers such as "illegal" / "unethical" that its SL list
  lacks. That is why the demo shows both readouts.
* **ALT-2 fails.** The base-geometry contrast has the "wrong" sign at the pre-registered layer and flips at each model's
  own layer. It explains only a small share of item-level variance.
* **C4 failure branch.** The ancestor (pt) direction stays near zero refusal across the α grid, like the random and
  language-ID controls, while each model's own direction (`own_pooled`) induces refusal. That is the positive control.